# Experimental RAG Project – Chunking Comparison

Submitted by:
- Noam Z.

Track:
Experimental / Research RAG – Track 2

Project topic: This project compares different chunking strategies in a simple RAG pipeline using educational documents about Artificial Intelligence and Information Retrieval.

## Project Type

This project follows Track 2: Experimental / Research RAG.

The goal of this project is to investigate how different chunking configurations affect retrieval quality in a RAG system.

The experiment compares three chunking configurations:

- Small chunks: 300 characters with 50 overlap
- Medium chunks: 500 characters with 100 overlap
- Large chunks: 800 characters with 150 overlap

All configurations use the same dataset, the same test questions, and the same TF-IDF retriever.

Only the chunk size and overlap are changed between the configurations.

In [9]:
from google.colab import files
import os
import shutil

os.makedirs("data", exist_ok=True)

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, os.path.join("data", filename))

print("Uploaded files to data folder:")
print(os.listdir("data"))

Saving doc1_machine_learning.txt to doc1_machine_learning.txt
Saving doc2_information_retrieval.txt to doc2_information_retrieval.txt
Saving doc3_rag.txt to doc3_rag.txt
Saving doc4_chunking.txt to doc4_chunking.txt
Saving doc5_embeddings.txt to doc5_embeddings.txt
Saving doc6_vector_databases.txt to doc6_vector_databases.txt
Uploaded files to data folder:
['doc4_chunking.txt', 'doc5_embeddings.txt', 'doc6_vector_databases.txt', 'doc1_machine_learning.txt', 'doc2_information_retrieval.txt', 'doc3_rag.txt']


## Step 1: Load the document collection

In this step, I load the text documents from the `data` folder.

Each text file represents one document in the knowledge base.  
These documents will later be split into chunks and used for retrieval in the RAG pipeline.

In [10]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "data"

documents = []

for filename in os.listdir(DATA_DIR):
    if filename.endswith(".txt"):
        file_path = os.path.join(DATA_DIR, filename)

        with open(file_path, "r", encoding="utf-8") as file:
            text = file.read()

        documents.append({
            "filename": filename,
            "text": text
        })

print(f"Loaded {len(documents)} documents.")

for doc in documents:
    print("-", doc["filename"])

Loaded 6 documents.
- doc4_chunking.txt
- doc5_embeddings.txt
- doc6_vector_databases.txt
- doc1_machine_learning.txt
- doc2_information_retrieval.txt
- doc3_rag.txt


## Step 2: Split documents into chunks

In this step, I split each document into smaller text chunks.

Chunking is important in RAG because the retrieval system usually searches over smaller text segments instead of full documents.

In this experiment, I compare three different chunking settings:
- Small chunks: 300 characters with 50 overlap
- Medium chunks: 500 characters with 100 overlap
- Large chunks: 800 characters with 150 overlap

In [11]:
def split_text_into_chunks(text, chunk_size, overlap):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks


chunking_configs = [
    {"name": "small_chunks", "chunk_size": 300, "overlap": 50},
    {"name": "medium_chunks", "chunk_size": 500, "overlap": 100},
    {"name": "large_chunks", "chunk_size": 800, "overlap": 150}
]


all_chunk_sets = {}

for config in chunking_configs:
    chunk_records = []

    for doc in documents:
        chunks = split_text_into_chunks(
            doc["text"],
            chunk_size=config["chunk_size"],
            overlap=config["overlap"]
        )

        for i, chunk in enumerate(chunks):
            chunk_records.append({
                "source_file": doc["filename"],
                "chunk_id": i,
                "text": chunk
            })

    all_chunk_sets[config["name"]] = chunk_records

    print(f"{config['name']}:")
    print(f"  Chunk size: {config['chunk_size']}")
    print(f"  Overlap: {config['overlap']}")
    print(f"  Total chunks: {len(chunk_records)}")
    print()

small_chunks:
  Chunk size: 300
  Overlap: 50
  Total chunks: 25

medium_chunks:
  Chunk size: 500
  Overlap: 100
  Total chunks: 18

large_chunks:
  Chunk size: 800
  Overlap: 150
  Total chunks: 12



The results show that smaller chunk sizes create more chunks, while larger chunk sizes create fewer chunks.

This is expected because small chunks divide the same documents into more text segments, while large chunks preserve more text in each segment.

## Step 3: Build a TF-IDF retriever for each chunking configuration

In this step, I build a separate TF-IDF retriever for each chunking configuration.

Each chunk is represented as a TF-IDF vector.  
When a user asks a question, the query is also converted into a TF-IDF vector, and cosine similarity is used to find the most relevant chunks.

This allows comparing how different chunk sizes affect retrieval results.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

retrievers = {}

for config_name, chunks in all_chunk_sets.items():
    chunk_texts = [chunk["text"] for chunk in chunks]

    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(chunk_texts)

    retrievers[config_name] = {
        "vectorizer": vectorizer,
        "tfidf_matrix": tfidf_matrix,
        "chunks": chunks
    }

    print(f"Built retriever for {config_name}")
    print("TF-IDF matrix shape:", tfidf_matrix.shape)
    print()

Built retriever for small_chunks
TF-IDF matrix shape: (25, 269)

Built retriever for medium_chunks
TF-IDF matrix shape: (18, 260)

Built retriever for large_chunks
TF-IDF matrix shape: (12, 255)



The TF-IDF matrix shape shows the number of chunks and the number of extracted textual features for each configuration.

Smaller chunks create more rows in the matrix because the documents are divided into more text segments.
Larger chunks create fewer rows because each chunk contains more text.

## Step 4: Define test questions and retrieve relevant chunks

In this step, I define several test questions for the RAG system.

For each question, the system retrieves the most relevant chunks from each chunking configuration using TF-IDF and cosine similarity.

This allows comparing whether small, medium, or large chunks return better context for answering the question.

In [13]:
test_questions = [
    {
        "question": "What is Retrieval-Augmented Generation?",
        "expected_topic": "RAG"
    },
    {
        "question": "Why is chunking important in RAG systems?",
        "expected_topic": "Chunking"
    },
    {
        "question": "What is the role of embeddings in semantic search?",
        "expected_topic": "Embeddings"
    },
    {
        "question": "How does an inverted index help in information retrieval?",
        "expected_topic": "Information Retrieval"
    },
    {
        "question": "What is a vector database used for?",
        "expected_topic": "Vector Databases"
    }
]


def retrieve_chunks(question, config_name, top_k=3):
    retriever = retrievers[config_name]

    vectorizer = retriever["vectorizer"]
    tfidf_matrix = retriever["tfidf_matrix"]
    chunks = retriever["chunks"]

    query_vec = vectorizer.transform([question])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []

    for idx in top_indices:
        results.append({
            "config": config_name,
            "question": question,
            "source_file": chunks[idx]["source_file"],
            "chunk_id": chunks[idx]["chunk_id"],
            "score": similarities[idx],
            "text": chunks[idx]["text"]
        })

    return results


for item in test_questions:
    question = item["question"]

    print("=" * 100)
    print("Question:", question)
    print("Expected topic:", item["expected_topic"])

    for config in chunking_configs:
        config_name = config["name"]

        print("\nChunking configuration:", config_name)

        results = retrieve_chunks(question, config_name, top_k=3)

        for rank, result in enumerate(results, start=1):
            print(f"\nRank {rank}")
            print("Source file:", result["source_file"])
            print("Chunk ID:", result["chunk_id"])
            print("Score:", round(result["score"], 4))
            print("Text:")
            print(result["text"][:500])
            print("-" * 80)

Question: What is Retrieval-Augmented Generation?
Expected topic: RAG

Chunking configuration: small_chunks

Rank 1
Source file: doc3_rag.txt
Chunk ID: 0
Score: 0.6958
Text:
Retrieval-Augmented Generation

Retrieval-Augmented Generation, also called RAG, is a method that combines information retrieval with text generation. Instead of asking a language model to answer only from its internal knowledge, the system first retrieves relevant documents or text chunks from an e
--------------------------------------------------------------------------------

Rank 2
Source file: doc3_rag.txt
Chunk ID: 4
Score: 0.3389
Text:
the chunking strategy, the retrieval method, and the language model used for generation.
--------------------------------------------------------------------------------

Rank 3
Source file: doc2_information_retrieval.txt
Chunk ID: 0
Score: 0.1573
Text:
Information Retrieval

Information retrieval is the process of finding relevant information from a large collection of docum

### Initial Retrieval Observation

The retrieval results show that for most questions, the highest-ranked chunk comes from the expected source document.

For example, the RAG question retrieves chunks from the RAG document, the chunking question retrieves chunks from the chunking document, and the embeddings question retrieves chunks from the embeddings document.

This shows that the TF-IDF retriever is able to find relevant chunks for the test questions.

However, the retrieved context is not identical across the different chunking configurations.  
Small chunks are more focused but sometimes incomplete, while larger chunks provide more context but may include extra information.

## Step 5: Evaluate retrieval quality

In this step, I evaluate the retrieval quality for each chunking configuration.

For each question and each chunking configuration, I check whether the top retrieved chunk came from the expected source document.

This gives a simple automatic evaluation score called Top-1 Accuracy.

A higher Top-1 Accuracy means that the retriever usually returns the correct document as the first result.

In [14]:
expected_files = {
    "What is Retrieval-Augmented Generation?": "doc3_rag.txt",
    "Why is chunking important in RAG systems?": "doc4_chunking.txt",
    "What is the role of embeddings in semantic search?": "doc5_embeddings.txt",
    "How does an inverted index help in information retrieval?": "doc2_information_retrieval.txt",
    "What is a vector database used for?": "doc6_vector_databases.txt"
}

evaluation_rows = []

for item in test_questions:
    question = item["question"]
    expected_file = expected_files[question]

    for config in chunking_configs:
        config_name = config["name"]

        results = retrieve_chunks(question, config_name, top_k=3)

        top_1_file = results[0]["source_file"]
        top_1_correct = top_1_file == expected_file

        top_3_files = [result["source_file"] for result in results]
        top_3_correct = expected_file in top_3_files

        evaluation_rows.append({
            "question": question,
            "expected_file": expected_file,
            "chunking_config": config_name,
            "top_1_file": top_1_file,
            "top_1_correct": top_1_correct,
            "top_3_files": ", ".join(top_3_files),
            "top_3_correct": top_3_correct
        })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df)

summary_df = evaluation_df.groupby("chunking_config").agg(
    top_1_accuracy=("top_1_correct", "mean"),
    top_3_accuracy=("top_3_correct", "mean")
).reset_index()

display(summary_df)

,question,expected_file,chunking_config,top_1_file,top_1_correct,top_3_files,top_3_correct
0,What is Retrieval-Augmented Generation?,doc3_rag.txt,small_chunks,doc3_rag.txt,True,"doc3_rag.txt, doc3_rag.txt, doc2_information_r...",True
1,What is Retrieval-Augmented Generation?,doc3_rag.txt,medium_chunks,doc3_rag.txt,True,"doc3_rag.txt, doc2_information_retrieval.txt, ...",True
2,What is Retrieval-Augmented Generation?,doc3_rag.txt,large_chunks,doc3_rag.txt,True,"doc3_rag.txt, doc3_rag.txt, doc2_information_r...",True
3,Why is chunking important in RAG systems?,doc4_chunking.txt,small_chunks,doc4_chunking.txt,True,"doc4_chunking.txt, doc3_rag.txt, doc3_rag.txt",True
4,Why is chunking important in RAG systems?,doc4_chunking.txt,medium_chunks,doc4_chunking.txt,True,"doc4_chunking.txt, doc3_rag.txt, doc3_rag.txt",True
5,Why is chunking important in RAG systems?,doc4_chunking.txt,large_chunks,doc4_chunking.txt,True,"doc4_chunking.txt, doc3_rag.txt, doc5_embeddin...",True
6,What is the role of embeddings in semantic sea...,doc5_embeddings.txt,small_chunks,doc5_embeddings.txt,True,"doc5_embeddings.txt, doc5_embeddings.txt, doc6...",True
7,What is the role of embeddings in semantic sea...,doc5_embeddings.txt,medium_chunks,doc5_embeddings.txt,True,"doc5_embeddings.txt, doc5_embeddings.txt, doc6...",True
8,What is the role of embeddings in semantic sea...,doc5_embeddings.txt,large_chunks,doc5_embeddings.txt,True,"doc5_embeddings.txt, doc6_vector_databases.txt...",True
9,How does an inverted index help in information...,doc2_information_retrieval.txt,small_chunks,doc2_information_retrieval.txt,True,"doc2_information_retrieval.txt, doc2_informati...",True


,chunking_config,top_1_accuracy,top_3_accuracy
0,large_chunks,1.0,1.0
1,medium_chunks,1.0,1.0
2,small_chunks,1.0,1.0


### Evaluation Results Explanation

The automatic evaluation shows that all three chunking configurations achieved perfect Top-1 and Top-3 accuracy.

This means that for every test question, the first retrieved chunk came from the expected source document, and the expected document also appeared within the top 3 retrieved chunks.

However, this does not mean that all chunking configurations are equally good in practice.

The dataset is small and clean, and each question is strongly related to one specific document.  
Therefore, the automatic accuracy scores are not sensitive enough to show the qualitative differences between the retrieved chunks.

For this reason, I also compare the retrieved text qualitatively and examine whether the chunks are focused, complete, and useful as context for answer generation.

## Step 6: Save Output Files

In this step, I save the evaluation results and summary results into CSV files.

These files can be used as output files for reviewing the experiment results.

In [15]:
import os

os.makedirs("outputs", exist_ok=True)

evaluation_df.to_csv("outputs/evaluation_results.csv", index=False)
summary_df.to_csv("outputs/summary_results.csv", index=False)

print("Saved evaluation results to outputs/evaluation_results.csv")
print("Saved summary results to outputs/summary_results.csv")

Saved evaluation results to outputs/evaluation_results.csv
Saved summary results to outputs/summary_results.csv


## Step 7: Answer Generation from Retrieved Context

In this step, the system generates simple answers based on the retrieved chunks.

This project focuses mainly on the retrieval and chunking part of RAG.  
Therefore, the generation step is implemented as a simple context-based answer generation method.

The generated answer is based only on the retrieved chunks and does not use external knowledge.

In [16]:
def generate_answer(question, retrieved_results):
    best_result = retrieved_results[0]

    answer = f"""Question:
{question}

Generated answer based on the most relevant retrieved chunk:
{best_result["text"]}

Source file: {best_result["source_file"]}
Similarity score: {round(best_result["score"], 4)}
"""
    return answer


best_config = "medium_chunks"

print("Generated answers using the best chunking configuration:", best_config)

for item in test_questions:
    question = item["question"]

    retrieved_results = retrieve_chunks(question, best_config, top_k=3)
    answer = generate_answer(question, retrieved_results)

    print("=" * 100)
    print(answer)

Generated answers using the best chunking configuration: medium_chunks
Question:
What is Retrieval-Augmented Generation?

Generated answer based on the most relevant retrieved chunk:
Retrieval-Augmented Generation

Retrieval-Augmented Generation, also called RAG, is a method that combines information retrieval with text generation. Instead of asking a language model to answer only from its internal knowledge, the system first retrieves relevant documents or text chunks from an external knowledge source.

A typical RAG pipeline includes document loading, chunking, indexing, retrieval, and answer generation. The documents are divided into smaller chunks, and each chunk is stor

Source file: doc3_rag.txt
Similarity score: 0.5855

Question:
Why is chunking important in RAG systems?

Generated answer based on the most relevant retrieved chunk:
Chunking in RAG Systems

Chunking is the process of dividing long documents into smaller text segments. In RAG systems, chunking is important because

### Answer Generation Observation

The generated answers are based on the most relevant retrieved chunk from the selected best configuration, which is `medium_chunks`.

This simple generation method does not use an external LLM. Instead, it extracts the most relevant retrieved context and presents it as the answer.

The answers are grounded in the retrieved documents and do not use external knowledge.

Some generated answers end with incomplete words because the current chunking method splits the text by character length. This is one of the limitations of simple character-based chunking.

## Step 8: Qualitative Analysis and Conclusions

The automatic evaluation shows that all three chunking configurations achieved perfect Top-1 and Top-3 accuracy on the selected test questions.

However, the qualitative analysis of the retrieved chunks shows differences between the configurations:

- Small chunks often return very focused results, but they may cut sentences or lose context.
- Medium chunks usually provide a good balance between precision and context.
- Large chunks preserve more context, but may include additional information that is not directly relevant to the question.

Based on these results, the medium chunking configuration appears to be the best choice for this small RAG experiment, because it provides enough context without returning overly long chunks.

## Step 9: Limitations

This experiment was performed on a small and clean dataset of six short educational documents.

Because the dataset is small and each question is strongly connected to one specific document, all chunking configurations achieved perfect automatic accuracy.

This means that the automatic metrics alone are not sensitive enough to show all differences between the chunking configurations.

In a larger and noisier dataset, the differences between chunking strategies would probably become more significant.

Future work could include more documents, longer documents, noisier text, more ambiguous questions, and additional retrieval methods such as BM25 or embeddings.

## Final Summary

In this project, I implemented a simple experimental Retrieval-Augmented Generation (RAG) pipeline.

The goal of the project was to compare different chunking configurations and examine how chunk size and overlap affect retrieval quality.

The pipeline includes:
1. Loading a small document collection.
2. Splitting the documents into chunks.
3. Building a TF-IDF retriever for each chunking configuration.
4. Retrieving the most relevant chunks for each test question using cosine similarity.
5. Evaluating retrieval quality using Top-1 Accuracy and Top-3 Accuracy.
6. Generating simple answers based on the retrieved context.

The experiment compared three chunking configurations:
- Small chunks: 300 characters with 50 overlap.
- Medium chunks: 500 characters with 100 overlap.
- Large chunks: 800 characters with 150 overlap.

The automatic evaluation showed that all configurations achieved 1.0 Top-1 Accuracy and 1.0 Top-3 Accuracy on the selected test questions. This means that the expected source document was retrieved successfully for every question.

However, the qualitative analysis showed that the retrieved chunks were not identical across the configurations. Small chunks were usually focused, but sometimes too short or incomplete. Large chunks preserved more context, but sometimes included additional information that was not directly needed for answering the question. Medium chunks provided the best balance between focused retrieval and enough useful context.

Therefore, for this small RAG experiment, the medium chunking configuration was selected as the most balanced option.

This experiment also has limitations. The dataset is small and clean, and each test question is strongly related to one specific document. Because of this, the automatic accuracy scores were perfect for all configurations. In a larger or noisier dataset, the differences between chunking strategies would probably become more significant.